<a href="https://colab.research.google.com/github/krutmanis95/StartSchool/blob/main/1_1_Linear_regression_wind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset

In this task we will use *wind energy* related datasets. The goal is to predict *energy production* by wind turbine based on meteorological conditions.

Let's download the dataset...

In [ ]:
!gdown https://drive.google.com/uc?id=1gcd9sv8NKkLTjm5TCIqcIWymzy3maSLL
!unzip -o wind_processed.zip

The dataset consists of information about *meteorological conditions* in particular place, as well as *energy produced* by two wind turbines located at the same place. There are two folders:

- `nora3` folder contains meteorological observations for ~25 years from [NORA3](https://help.emd.dk/mediawiki/index.php/NORA3) dataset.
- `aves` folder contains energy yield of the turbines for several years.

## Task: Load and join datasets

Both datasets are well structured and are easy to work with.

1. Load all `nora3/*.csv` files into *Pandas* the data frame
    * read each CSV file (`pd.read_csv`);
    * use *timestamp* as an index (specify `index_col` parameter);
    * load only columns related to *temperature*, *relative humidity* and *wind speed* and *direction* at 50 meters (specify `usecols` parameter);
    * parse *timestamp* as datetime (specify `parse_dates` parameter);
    * concatenate loaded dataset into single data frame (e.g. `nora = pd.concat(...)`);
    * hint: sort dataset by *timestamp* (e.g. `sort_index`);
    * hint: use `glob` module to get the list of files and iterate over it;
    * hint: use `nora.describe` to validate loaded data.

2. Load all `aves/*.csv` files into *Pandas* the data frame
    * similarly to 1st point, read and parse each CSV file;
    * concatenate loaded dataset into single data frame (e.g. `aves = pd.concat(...)`).

3. Join both datasets
    * use `join` function with `inner` operation mode (`merge` with proper parameters will do the trick as well);
    * note: as both data frames are indexed by *timestamp*, the join is trivial;
    * hint: use `validate` parameter to ensure *one-to-one* join;
    * store the result in separate variable (e.g. `data`).

* Hint: Use separate code blocks for each point for faster prototyping.

In [ ]:
import pandas as pd
import glob

# TODO: your code goes here

## Task: Explore the dataset

Before building any ML model the good practice is to get understanding of your data and find if there any unexpected values (e.g. missing values, or extremely low/high values, measurement errors, etc).

One of the simplest approaches to investigate the dataset is to display [box plot](https://en.wikipedia.org/wiki/Box_plot): it graphically shows main statistical properties of the values (median, quartiles and outliers).

Another powerful approach is to analyze [correlation](https://en.wikipedia.org/wiki/Correlation) between data features. Visually it can be presented as a scatter plot.

Also plain simple timeline plots might be useful.

1. Create box plots for each column in the dataset
    * use *Pandas* provided API (e.g. `data.plot.box`);
    * hint: specify `subplots`, `grid` and/or other parameters for better results.

2. Show correlation between *wind speed* and *energy production* for each turbine
    * create a figure with two columns (`plt.subplots`);
    * hint: use `sharex` and/or `sharey` parameter to share tick labels between subplots;
    * render `scatter` plot for each turbine;
    * provide *wind speed* values as `x` parameter;
    * provide *energy yield* values as `y` parameter;
    * hint: use low `alpha` and `s` (size) values for better results.

3. Show energy production timeline
    * create a figure with two rows (`plt.subplots`)
    * hint: use `sharex` and/or `sharey` parameters to share tick labels between subplots;
    * render `scatter` plot for each turbine;
    * provide *timestamp* values (`index`) as `x` parameter;
    * provide *energy yield* values as `y` parameter;
    * hint: use low `alpha` and `s` (size) values for better results;
    * render `step` plot of *monthly average* energy yield;
    * hint: use `x.resample("MS").mean()` to aggregate over *timestamp* (`MS` stands for "Month Start");
    * hint: use `color`, `label` and other parameters if needed.
    
* Hint: use separate code blocks for each figure, or `plt.show()` to render multiple figures at once.
* Hint: make sure the figures "looks nice" (appropriate colors, text sizes, axis labels, grids, legends, etc).

In [ ]:
import matplotlib.pyplot as plt

# for nice subplots
plt.rcParams["figure.constrained_layout.use"] = True

# TODO: your code goes here

# Linear regression model

Now your task is to train **linear regression model** for the wind dataset. The ultimate goal is to provide *meteorological parameters* to the model and get *energy yield* prediction.

The task of linear regression is to find such values of $w_n$ and $b$ variables, so that following equation describes the data as close as possible.

$$\hat{y} = w_1x_1 + w_2x_2 + w_3x_3 + ... + b$$

The inputs $x_n$ for the model are various *meteorological parameters*, and the expected output $\hat{y}$ is *energy yield* of each turbine.

> Note: In some sense you are creating two linear regressions (one for each turbine). However from code perspective there is no need for separation, you can create single model with two outputs.

## Train and test datasets
First of all, we have one big dataset. If we use all available examples for training the model (fitting the variables), we won't be able to check how well our model is trained.

Thus usual approach is to "hide" part of examples from the model during the training, and use it later for testing the model.

Also as a naming convention it is recommended to have separate objects for input values and expected output values. Input values are usually designated as uppercase `X`, output values as lowercase `y`. Keep in mind that both `X` and `y` variables might be tables containing multiple columns.

Also different "flavors" of these variables are often used, e.g. `X_test`, `y_train`, `X_batch`, `y_pred`, etc.

## Task: Train linear regression model

In previous stage we have explored the dataset. One of the problems you might discovered is that there are records with zero or negative *energy yield* values. These might be due to hardware failure or maintenance on the turbines. Such records are *not typical* for the turbines, thus, should be removed from the dataset.

1. Clean-up the dataset
    * keep only records with valid *energy yield* values (e.g. above 5 kWh);
    * store results as separate variable (e.g. `data_clean`);
    * hint: use *Pandas* syntax to filter records (e.g. `df[df["A"] >= 5]`).

2. Define variables for input and output values
    * follow naming conventions, e.g. use `X_data` for *meteorological parameters* , and `y_data` for the *energy yield*.
    * store all examples (rows) in these variables;
    * hint: use *Pandas* syntax to fetch specific columns (e.g. `X = df[["A", "B", "C"]]`).

3. Split the dataset into `train` and `test` parts.
    * use `sklearn.model_selection.train_test_split` utility function;
    * use `20%` of samples for testing;
    * store results in separate variables, e.g. `X_train`, `X_test`, `y_train`, `y_test`.

4. Create linear regression model
    * create new `tf.keras.Sequential` model;
    * add `Input` layer with appropriate `shape`;
    * add `Dense` layer with appropriate number of `units`;
    * `compile` the model with `Adam` optimizer, `MeanSquaredError` loss and `r2_score` metric.

5. Train the model
    * use `fit` method to train the model;
    * provide `train` dataset;
    * specify `50` epochs.

6. Evaluate the model
    * print *metric* values for `test` dataset (e.g. use `evaluate` method);
    * use `predict` method to get model outputs for `test` dataset;
    * create `scatter` plot comparing true values from `y_test` and the values predicted by the model;
  
* Make conclusions about the performance of the model. How good is it in your opinion?
* Hint: refer to the documentation of a specific function/library if you are not sure what kind of parameters it uses or results produce.
* Hint: don't use additional libraries without propper understanding (professor will ask for your justification :).

In [ ]:
import tensorflow as tf
# Note: we can import specific functions from the library
from sklearn.model_selection import train_test_split

# TODO: add your code here